In [ ]:
import pandas as pd
import requests
from io import StringIO
import certifi
import matplotlib.pyplot as plt
import mesa
import numpy as np

# Module 2: Population Epidemiological Simulation

### Epidemiological simulator that uses, as input, the fitness of variants predicted in Module 1 to simulate the spatial and temporal spread of the virus in the population and to test intervention scenarios.

- A simulation model that represents individuals, infections, and transmission over time.

- Use at least one parameter coming from Module 1 (for example, fitness, implicit R₀, probability of infection per contact, duration of infection, or severity) to differentiate variants.

- Allow the simulation of at least two public health scenarios (for example: no measures vs. masks; no measures vs. distancing; low vs. high vaccination coverage) and compare the results.

**Mesa** and **AgentPy** are agent-based modeling libraries in Python, ideal for representing individuals and interactions in an epidemiological simulation. Within the scope of this work, they allow the fitness predicted in Module 1 to be used as a parameter that controls transmission probability and infection dynamics, making explicit the link between molecular properties of variants and their spread at the population level.


| **Aspect** | **Mesa** | **AgentPy** |
|------|------|---------|
| Main focus | Models + interactive web visualization | Models + numerical experiments in Jupyter |
| Maturity / ecosystem | More mature, more examples and tutorials | Lighter, but well designed for notebooks |
| Visualization | Browser-based interface (Solara), ready-made components | Integration with ipysimulate / Jupyter |
| Agent / space management | AgentSet, multiple space classes | AgentList, Space / Grid / Network |
| Parameter sweeps | Possible, but less “native” | ap.Experiment is a core feature |
| Learning curve | Slightly more “framework-heavy” | Minimalist API, well suited for notebooks |


For the empirical component of this work, the **COVID-19 dataset from the Our World in Data (OWID)** platform will be used. This platform compiles standardized daily time series of multiple epidemiological indicators for virtually all countries. The dataset includes, among others, *confirmed cases*, *deaths*, *hospitalizations*, *testing data*, and *vaccination metrics*, organized with one row per country and date, which allows the temporal evolution of the pandemic to be directly related to demographic characteristics and health system features.

In the context of this project, these data serve as a realistic basis for calibrating and evaluating the simulation model developed in Module 2. This enables, for example, comparisons between simulated incidence or mortality curves and the trajectories observed in selected countries, as well as the exploration of intervention scenarios inspired by real-world patterns of vaccination and public health responses.

## 1. Load and preprocess the OWID dataset

We load the latest cleaned dataset from Our World in Data (OWID) and run basic exploratory checks:

- number of rows and columns

- column types and missing values

- distribution of daily cases, deaths, and vaccinations over time

- overview of countries and global trends

In [2]:
url = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"

# Use certifi to verify SSL
r = requests.get(url, verify=certifi.where())
csv_data = StringIO(r.text)

df = pd.read_csv(csv_data)

In [3]:
print(f"Dataset shape: {df.shape}")
print("\nColumn types:\n", df.dtypes)
print("\nMissing values per column:\n", df.isna().sum())

# Top 5 rows
df.head()

Dataset shape: (429435, 67)

Column types:
 iso_code                                       str
continent                                      str
location                                       str
date                                           str
total_cases                                float64
                                            ...   
population                                   int64
excess_mortality_cumulative_absolute       float64
excess_mortality_cumulative                float64
excess_mortality                           float64
excess_mortality_cumulative_per_million    float64
Length: 67, dtype: object

Missing values per column:
 iso_code                                        0
continent                                   26525
location                                        0
date                                            0
total_cases                                 17631
                                            ...  
population                              

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-01-06,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-01-07,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-01-08,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-01-09,0.0,0.0,NaN,0.0,0.0,NaN,...,NaN,37.75,0.5,64.83,0.51,41128772,NaN,NaN,NaN,NaN


Before diving into analysis, we check the structure of the dataset:
- **Shape**: number of rows and columns, to understand dataset size
- **Columns**: the variables included, such as cases, deaths, tests, vaccinations, and location information
- **Missing values**: percentage of missing data per column, to identify gaps and potential preprocessing needs

In [4]:
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nMissing values (%):\n", df.isna().mean().round(3)*100)

Shape: (429435, 67)

Columns:
 ['iso_code', 'continent', 'location', 'date', 'total_cases', 'new_cases', 'new_cases_smoothed', 'total_deaths', 'new_deaths', 'new_deaths_smoothed', 'total_cases_per_million', 'new_cases_per_million', 'new_cases_smoothed_per_million', 'total_deaths_per_million', 'new_deaths_per_million', 'new_deaths_smoothed_per_million', 'reproduction_rate', 'icu_patients', 'icu_patients_per_million', 'hosp_patients', 'hosp_patients_per_million', 'weekly_icu_admissions', 'weekly_icu_admissions_per_million', 'weekly_hosp_admissions', 'weekly_hosp_admissions_per_million', 'total_tests', 'new_tests', 'total_tests_per_thousand', 'new_tests_per_thousand', 'new_tests_smoothed', 'new_tests_smoothed_per_thousand', 'positive_rate', 'tests_per_case', 'tests_units', 'total_vaccinations', 'people_vaccinated', 'people_fully_vaccinated', 'total_boosters', 'new_vaccinations', 'new_vaccinations_smoothed', 'total_vaccinations_per_hundred', 'people_vaccinated_per_hundred', 'people_fully_v

OWID’s dataset contains entries for aggregates like “World”, continents, and regions, which have ISO codes longer or shorter than three letters.
To focus our analysis on actual countries, we filter rows where the ISO code has exactly 3 characters:

In [6]:
countries = df[df["iso_code"].str.len() == 3]

**Best Countries for Our Study**

| **Country** | **Why use it** | **Example of possible analysis** |
|--------|------------|------------------------------|
| *Portugal* | Very complete national data (DGS → OWID), clear epidemic waves, local context | Calibration using the 1st wave (Mar–May 2020) or the Delta wave (2021) |
| *Italy* | Intense first wave in Europe, detailed data available since Feb 2020 | Validate early simulations without vaccination |
| *United Kingdom* | Exceptional data (testing, vaccination, hospitalizations), multiple waves | Compare vaccinated vs. non-vaccinated scenarios |
| *Spain* | Rich data, cultural proximity, well-defined peaks | Effect of lockdowns on disease spread |
| *New Zealand* | Elimination strategy, excellent data quality, strong contrast with Europe | Simulate “zero COVID” vs. free-spread waves |
| *Singapore* | High population density, Asian data, well-documented variant waves | Effect of variants in a dense population |
| *Sweden* | Controversial strategy (no strict lockdowns), useful for alternative scenarios | Comparison with Nordic neighbors |

**Practical Recommendation**

- *For an MVP:* Portugal + Italy (local data combined with a classic European reference).

- *For an extension:* add the United Kingdom (more variables available) and New Zealand (extreme contrast).

- *Time period:* 2020-01-22 to 2021-12-31 (period covering the main epidemic waves and the initial vaccination rollo



## 2. Define the output of Module 1 that will be used in Module 2

## 3. Choose a library for the ABM (Mesa or AgentPy) and implement:
- Agents (S/I/R individuals, with a variant attribute and transmission probability)
- Model (global parameters, scheduler, metrics logging)
- Scenarios (no measures vs. intervention)